In [1]:
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

import pandas as pd
pd.options.mode.chained_assignment = None

**Data Pre-Processing & Labeling**

In [2]:
df = pd.read_csv('../dataset/movie_reviews.csv')

# Remove any Null value rows and duplicates
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

df.info()

<class 'pandas.DataFrame'>
Index: 23820 entries, 0 to 25209
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   title    23820 non-null  str    
 1   content  23820 non-null  str    
 2   rating   23820 non-null  float64
dtypes: float64(1), str(2)
memory usage: 744.4 KB


In [3]:
df['full_text'] = df['title'] + ' ' + df['content']
df.info()

<class 'pandas.DataFrame'>
Index: 23820 entries, 0 to 25209
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      23820 non-null  str    
 1   content    23820 non-null  str    
 2   rating     23820 non-null  float64
 3   full_text  23820 non-null  str    
dtypes: float64(1), str(3)
memory usage: 930.5 KB


In [4]:
def sentiment_labeling(rating):
    polarity = ''
    
    if rating <= 4:
        polarity = 'negative'
    elif rating >= 7:
        polarity = 'positive'
    else:
        polarity = 'neutral'
        
    return polarity

In [5]:
results = df['rating'].apply(sentiment_labeling)
df['polarity'] = results
df['polarity'].value_counts()

polarity
positive    11995
negative     8501
neutral      3324
Name: count, dtype: int64

In [6]:
def cleaning_text(text):
    text = re.sub(r'[0-9]+', '', text) # remove numbers
    text = re.sub(r'[^\w\s]', '', text) # remove special characters
 
    text = text.replace('\n', ' ') # remove new lines
    text = text.translate(str.maketrans('', '', string.punctuation)) # remove punctuations
    text = text.strip(' ') # trimming
    return text
 
def casefolding_text(text):
    text = text.lower()
    return text
 
def tokenizing_text(text):
    text = word_tokenize(text)
    return text
 
def stopword_text(text):
    listStopwords = set(stopwords.words('english'))
    listStopwords.update(listStopwords)
    filtered = [word for word in text if word not in listStopwords]
    return filtered
 
def stemming_text(text):
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(word) for word in text]
    return stemmed

def lemmatizing_text(text):
    lemmer = WordNetLemmatizer()
    lemmatized = [lemmer.lemmatize(word) for word in text]
    return lemmatized
 
def to_sentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [7]:
clean_df = df
clean_df['clean_text'] = clean_df['full_text'].apply(cleaning_text)
clean_df['casefold_text'] = clean_df['clean_text'].apply(casefolding_text)
clean_df['tokenized_text'] = clean_df['casefold_text'].apply(tokenizing_text)
clean_df['stopword_text'] = clean_df['tokenized_text'].apply(stopword_text)

clean_df['stemmed_text'] = clean_df['stopword_text'].apply(stemming_text)
clean_df['lemmatized_text'] = clean_df['stopword_text'].apply(lemmatizing_text)
clean_df['final_full_text'] = clean_df['stopword_text'].apply(to_sentence)

In [8]:
clean_df.head()

,title,content,rating,full_text,polarity,clean_text,casefold_text,tokenized_text,stopword_text,stemmed_text,lemmatized_text,final_full_text
0,The death of a franchise.,The Last Jedi is a well-made film; it's visual...,6.0,The death of a franchise. The Last Jedi is a w...,neutral,The death of a franchise The Last Jedi is a we...,the death of a franchise the last jedi is a we...,"[the, death, of, a, franchise, the, last, jedi...","[death, franchise, last, jedi, wellmade, film,...","[death, franchis, last, jedi, wellmad, film, v...","[death, franchise, last, jedi, wellmade, film,...",death franchise last jedi wellmade film visual...
1,Worst Star Wars movie,Why is Luke being annihilated like this? After...,6.0,Worst Star Wars movie Why is Luke being annihi...,neutral,Worst Star Wars movie Why is Luke being annihi...,worst star wars movie why is luke being annihi...,"[worst, star, wars, movie, why, is, luke, bein...","[worst, star, wars, movie, luke, annihilated, ...","[worst, star, war, movi, luke, annihil, like, ...","[worst, star, war, movie, luke, annihilated, l...",worst star wars movie luke annihilated like nu...
2,It's bad. I don't know why people defend it.,I really want to sit across from Rian Johnson ...,6.0,It's bad. I don't know why people defend it. I...,neutral,Its bad I dont know why people defend it I rea...,its bad i dont know why people defend it i rea...,"[its, bad, i, dont, know, why, people, defend,...","[bad, dont, know, people, defend, really, want...","[bad, dont, know, peopl, defend, realli, want,...","[bad, dont, know, people, defend, really, want...",bad dont know people defend really want sit ac...
4,Amazed by the negativity,I walked out of this movie with six friends at...,9.0,Amazed by the negativity I walked out of this ...,positive,Amazed by the negativity I walked out of this ...,amazed by the negativity i walked out of this ...,"[amazed, by, the, negativity, i, walked, out, ...","[amazed, negativity, walked, movie, six, frien...","[amaz, neg, walk, movi, six, friend, us, wire,...","[amazed, negativity, walked, movie, six, frien...",amazed negativity walked movie six friends us ...
5,Almost Garbage,"""Star Wars: The Last Jedi"" was a huge disappoi...",6.0,"Almost Garbage ""Star Wars: The Last Jedi"" was ...",neutral,Almost Garbage Star Wars The Last Jedi was a h...,almost garbage star wars the last jedi was a h...,"[almost, garbage, star, wars, the, last, jedi,...","[almost, garbage, star, wars, last, jedi, huge...","[almost, garbag, star, war, last, jedi, huge, ...","[almost, garbage, star, war, last, jedi, huge,...",almost garbage star wars last jedi huge disapp...


In [9]:
clean_df = clean_df[clean_df['polarity'] != 'neutral']

# neutral_df = clean_df[clean_df['polarity'] == 'neutral'].head(3300)
positive_df = clean_df[clean_df['polarity'] == 'positive'].head(8000)
negative_df = clean_df[clean_df['polarity'] == 'negative'].head(8000)

clean_df = pd.concat([positive_df, negative_df])

clean_df['polarity'].value_counts()

polarity
positive    8000
negative    8000
Name: count, dtype: int64

**Feature Extraction**

In [10]:
print(clean_df['stemmed_text'].head())
print(clean_df['lemmatized_text'].head())

4     [amaz, neg, walk, movi, six, friend, us, wire,...
6     [beauti, humor, magic, star, war, movi, ive, s...
10    [forc, properli, awaken, th, chapter, newli, r...
14    [wtf, wrong, peopl, good, movi, wont, post, sp...
16    [open, mind, father, star, war, start, say, yo...
Name: stemmed_text, dtype: object
4     [amazed, negativity, walked, movie, six, frien...
6     [beautiful, humorous, magical, star, war, movi...
10    [force, properly, awaken, th, chapter, newly, ...
14    [wtf, wrong, people, good, movie, wont, post, ...
16    [open, mind, father, star, war, start, saying,...
Name: lemmatized_text, dtype: object


In [11]:
# TF-IDF
tfidf_model = TfidfVectorizer(
    max_features=100_000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

In [12]:
def convert_polarity(polarity):
    return 1 if polarity == 'positive' else 0

In [13]:
X = clean_df['final_full_text']
Y = clean_df['polarity'].apply(convert_polarity)

In [14]:
# Random Forest Dataset
rm_X_train_raw, rm_X_test_raw, rm_Y_train, rm_Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
rm_X_train = tfidf_model.fit_transform(rm_X_train_raw)
rm_X_test = tfidf_model.transform(rm_X_test_raw)

In [15]:
random_forest = RandomForestClassifier(n_estimators=200)
random_forest.fit(rm_X_train, rm_Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [16]:
rf_predictions = random_forest.predict(rm_X_test)
print(classification_report(rm_Y_test, rf_predictions))

              precision    recall  f1-score   support

           0       0.89      0.86      0.88      2388
           1       0.87      0.90      0.88      2412

    accuracy                           0.88      4800
   macro avg       0.88      0.88      0.88      4800
weighted avg       0.88      0.88      0.88      4800



In [17]:
def predict_sentiment(text, rm_model, tfidf_model):
    cleaned = cleaning_text(text)
    casefolded = casefolding_text(cleaned)
    tokens = tokenizing_text(casefolded)
    tokens = stopword_text(tokens)
    tokens = lemmatizing_text(tokens)
    
    processed_text = ' '.join(tokens)
    
    tfidf_features = tfidf_model.transform([processed_text])
    
    prediction = rm_model.predict(tfidf_features)[0]
    label = 'positive' if prediction == 1 else 'negative'
    
    return label

In [18]:
reviews = [
    "It wasn't bad, but I wouldn't watch it again.",
    "The acting was great but the plot made no sense.",
    "I expected to hate it but ended up really enjoying it.",
    "Not the worst movie I've seen, but far from the best.",
    "This was an absolute masterpiece from start to finish, I couldn't look away for a second.",
    "A complete waste of time, the plot made no sense and the acting was wooden.",
    "It was okay, nothing special but not terrible either.",
    "The visuals were stunning but the story dragged on way too long.",
    "I went in with low expectations and ended up pleasantly surprised."
]

for idx, r in enumerate(reviews):
    label = predict_sentiment(r, random_forest, tfidf_model)
    print(f"{idx + 1}. {r}\n  -> {label}\n")

1. It wasn't bad, but I wouldn't watch it again.
  -> negative

2. The acting was great but the plot made no sense.
  -> negative

3. I expected to hate it but ended up really enjoying it.
  -> negative

4. Not the worst movie I've seen, but far from the best.
  -> negative

5. This was an absolute masterpiece from start to finish, I couldn't look away for a second.
  -> negative

6. A complete waste of time, the plot made no sense and the acting was wooden.
  -> negative

7. It was okay, nothing special but not terrible either.
  -> negative

8. The visuals were stunning but the story dragged on way too long.
  -> negative

9. I went in with low expectations and ended up pleasantly surprised.
  -> positive

